# SMB leads analysis: Udyam Digital
Synthetic data, Apr-Jun 2026. Small cells are flagged: a rate on under 30 activated customers is a lead to check, not a finding.

In [1]:
import pandas as pd, numpy as np
from scipy.stats import chi2_contingency, spearmanr
pd.set_option('display.width',200,'display.max_columns',30)
dates=['created_date','contacted_date','needs_assessed_date','quote_accepted_date','kyc_complete_date','activated_date']
d=pd.read_csv('../data/smb_leads.csv',parse_dates=dates)
d['act']=d.outcome=='Activated'; d['churn']=d.churned_within_6_months.map({'Y':1,'N':0})
a=d[d.act]
order=['Lead','Contacted','Needs assessed','Quote accepted','KYC complete','Activated']
vc=d.stage_reached.value_counts(); funnel=[int(vc[order[i:]].sum()) for i in range(6)]
assert funnel==[5000,3406,1974,861,588,368]; print(funnel)
print('CAC per activation', round(d.acquisition_cost_inr.sum()/d.act.sum()), '| days', round(a.days_lead_to_activation.mean(),1),'| chases', round(a.customer_follow_up_calls.mean(),2))

[5000, 3406, 1974, 861, 588, 368]
CAC per activation 15053 | days 17.6 | chases 3.01


## By segment, source, GSTIN, tier

In [2]:
def table(g):
    t=d.groupby(g).agg(leads=('act','size'),activated=('act','sum'),rate=('act','mean'),spend=('acquisition_cost_inr','sum'),value=('monthly_bundle_value_inr','mean'),churn=('churn','mean'),nps=('nps_score','mean'))
    t['cac']=t.spend/t.activated; t['payback_months']=t.cac/t.value
    return t.drop(columns='spend').round(3)
for g in ['segment','lead_source','has_gstin','city_tier','current_internet']: display(table(g))

,leads,activated,rate,value,churn,nps,cac,payback_months
segment,,,,,,,,
Micro retail & services,2709,110,0.041,2504.818,0.227,7.045,20007.091,7.987
Professional offices,1247,138,0.111,6795.870,0.138,6.920,10694.710,1.574
Small manufacturers,289,37,0.128,19979.730,0.081,6.459,15324.324,0.767
Small multi-site businesses,755,83,0.110,31233.494,0.108,6.699,15614.337,0.500


,leads,activated,rate,value,churn,nps,cac,payback_months
lead_source,,,,,,,,
Channel partner,1010,81,0.080,10359.136,0.185,6.753,20850.988,2.013
Field sales,1720,125,0.073,12965.920,0.096,6.800,24334.960,1.877
Inbound call,504,48,0.095,12508.542,0.146,6.646,5113.333,0.409
Referral,413,61,0.148,11076.557,0.246,7.115,4098.689,0.370
Website form,995,43,0.043,14834.186,0.116,7.070,6351.628,0.428
WhatsApp campaign,358,10,0.028,17121.000,0.200,7.100,4026.000,0.235


,leads,activated,rate,value,churn,nps,cac,payback_months
has_gstin,,,,,,,,
N,1466,30,0.020,6158.333,0.233,7.10,41727.667,6.776
Y,3534,338,0.096,12900.118,0.145,6.84,12685.828,0.983


,leads,activated,rate,value,churn,nps,cac,payback_months
city_tier,,,,,,,,
1,2578,221,0.086,12223.801,0.158,6.919,13479.638,1.103
2,2422,147,0.061,12541.020,0.143,6.776,17419.320,1.389


,leads,activated,rate,value,churn,nps,cac,payback_months
current_internet,,,,,,,,
Business broadband,1433,133,0.093,16223.233,0.083,6.835,14524.135,0.895
Home broadband,1250,95,0.076,5448.737,0.221,6.895,12582.526,2.309
Leased line,173,26,0.150,29045.385,0.115,6.654,14604.231,0.503
Mixed per site,221,28,0.127,29543.929,0.036,6.393,14766.429,0.500
Mobile hotspot,1799,81,0.045,3392.593,0.247,7.160,18714.321,5.516


## Drop reasons per stage

In [3]:
x=d[d.outcome=='Dropped'].groupby(['stage_reached','drop_reason']).size().rename('n').reset_index()
x['share_of_stage']=(x.n/x.groupby('stage_reached').n.transform('sum')).round(3)
display(x.sort_values(['stage_reached','n'],ascending=[True,False]))

,stage_reached,drop_reason,n,share_of_stage
2,Contacted,Owner had no time for a visit,531,0.371
0,Contacted,Locked in with current provider,325,0.227
3,Contacted,Visit not scheduled by sales,296,0.207
1,Contacted,Only wanted a mobile plan,280,0.196
6,KYC complete,Installation delayed beyond 21 days,98,0.445
4,KYC complete,Customer cancelled before install,53,0.241
7,KYC complete,Site not feasible for fibre,41,0.186
5,KYC complete,First payment failed,28,0.127
9,Lead,Unreachable after 3 attempts,805,0.505
8,Lead,Not interested on first call,470,0.295


## Signal tests

In [4]:
def chi(col,df,y):
    t=pd.crosstab(df[col],df[y]); return round(chi2_contingency(t)[1],5)
print('follow-up calls vs churn, Spearman', tuple(round(v,4) for v in spearmanr(a.customer_follow_up_calls,a.churn)))
print('follow-up calls vs NPS, Spearman', tuple(round(v,4) for v in spearmanr(a.customer_follow_up_calls,a.nps_score)))
display(a.groupby('customer_follow_up_calls').agg(n=('churn','size'),churn=('churn','mean'),nps=('nps_score','mean')).round(3))
print('source vs churn p', chi('lead_source',a,'churn'),'| segment vs churn p',chi('segment',a,'churn'),'| hotspot vs churn p',chi('current_internet',a,'churn'))
print('source vs activation p',chi('lead_source',d,'act'),'| gstin vs activation p',chi('has_gstin',d,'act'))
k=d[d.kyc_complete_date.notna()].copy(); k['install_days']=(k.activated_date-k.kyc_complete_date).dt.days
c=k.groupby('city').agg(kyc=('act','size'),activated=('act','sum'),rate=('act','mean'),install=('install_days','mean'),delay_drops=('drop_reason',lambda s:(s=='Installation delayed beyond 21 days').sum())).round(2)
display(c.sort_values('rate')); print('city vs post-KYC activation p',chi('city',k,'act'))
b=a.assign(band=pd.cut(a.days_lead_to_activation,[0,14,21,60],labels=['<=14','15-21','>21']))
display(b.groupby('band',observed=True).agg(n=('nps_score','size'),nps=('nps_score','mean'),churn=('churn','mean')).round(2))

follow-up calls vs churn, Spearman (np.float64(0.1364), np.float64(0.0088))
follow-up calls vs NPS, Spearman (np.float64(-0.2337), np.float64(0.0))


,n,churn,nps
customer_follow_up_calls,,,
1,34,0.059,7.471
2,84,0.143,7.190
3,137,0.117,6.891
4,70,0.200,6.657
5,43,0.279,5.977


source vs churn p 0.12997 | segment vs churn p 0.05232 | hotspot vs churn p 0.00184
source vs activation p 0.0 | gstin vs activation p 0.0


,kyc,activated,rate,install,delay_drops
city,,,,,
Lucknow,23,8,0.35,6.38,7
Bhopal,27,10,0.37,4.90,4
Vadodara,20,8,0.40,5.62,4
Coimbatore,20,9,0.45,5.89,6
Indore,20,9,0.45,8.33,5
Visakhapatnam,19,10,0.53,6.30,5
Patna,19,10,0.53,5.70,5
Nagpur,17,9,0.53,4.11,4
Mysuru,15,8,0.53,8.12,4


city vs post-KYC activation p 0.00135


,n,nps,churn
band,,,
<=14,86,7.22,0.16
15-21,222,6.96,0.14
>21,60,5.98,0.17


## What the fixes could be worth
Assumption, not data: the digital journey recovers **half** of the drops it directly addresses. Recovered leads then convert at the observed downstream rates.

In [5]:
dr=d.drop_reason.value_counts()
r=[('Contacted to assessed: no visit needed',dr['Owner had no time for a visit']+dr['Visit not scheduled by sales'],588/861*368/588*861/1974),
   ('Assessed to quote: instant, clear quote',dr['Quote too complex to understand']+dr['Quote took too long to arrive'],368/861),
   ('Quote to KYC: paperwork and mismatches',dr['Found paperwork too long']+dr['Document details did not match'],368/588)]
s=pd.DataFrame(r,columns=['fix','drops','downstream']); s['extra_activations']=(s.drops*.5*s.downstream).round(0)
display(s); tot=int(s.extra_activations.sum()); print('activations',368,'->',368+tot,'=',round((368+tot)/5000*100,1),'%')

,fix,drops,downstream,extra_activations
0,Contacted to assessed: no visit needed,827,0.186424,77.0
1,"Assessed to quote: instant, clear quote",385,0.427410,82.0
2,Quote to KYC: paperwork and mismatches,126,0.625850,39.0


activations 368 -> 566 = 11.3 %


## Propensity model
Profile fields known at lead creation only. Drop reason, stage dates, follow-up calls, bundle and acquisition cost are excluded as leakage.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
cat=['lead_source','segment','city_tier','has_gstin','has_udyam','annual_turnover_band','current_internet','current_provider','preferred_language','contact_role']
num=['employee_count','num_sites','monthly_telecom_spend_inr']
X=d[cat+num].copy(); X[cat]=X[cat].astype(str).fillna('NA'); y=d.act.astype(int)
pipe=make_pipeline(ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),cat),('n',StandardScaler(),num)]),LogisticRegression(max_iter=2000,class_weight='balanced'))
print('5-fold AUC', cross_val_score(pipe,X,y,cv=5,scoring='roc_auc').mean().round(3))
tr,te,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=7); pipe.fit(tr,ytr); pr=pipe.predict_proba(te)[:,1]
print('holdout AUC',round(roc_auc_score(yte,pr),3)); top=pd.Series(yte.values,index=pr).sort_index(ascending=False); n=len(top)//5
print('top-quintile activation',round(top.iloc[:n].mean(),3),'vs base',round(yte.mean(),3))
names=pipe[0].get_feature_names_out(); w=pd.Series(pipe[1].coef_[0],index=names).sort_values()
display(pd.concat([w.tail(8),w.head(5)]).round(2))

5-fold AUC 0.709
holdout AUC 0.735
top-quintile activation 0.148 vs base 0.074


c__segment_Small manufacturers             0.24
c__current_internet_Mixed per site         0.27
c__current_provider_Accel (mobile only)    0.28
c__lead_source_Channel partner             0.28
c__preferred_language_Malayalam            0.31
c__lead_source_Inbound call                0.38
c__has_gstin_Y                             0.52
c__lead_source_Referral                    0.87
c__lead_source_WhatsApp campaign          -1.06
c__has_gstin_N                            -0.65
c__lead_source_Website form               -0.54
c__segment_Micro retail & services        -0.49
c__preferred_language_Tamil               -0.49
dtype: float64